In [ ]:
import pandas as pd
from pathlib import Path

# Configuration
input_file = Path("Data/processed/clubs/clubs_master.csv")
output_audit_file = Path("Data/processed/clubs/clubs_audit_report.csv")

def audit_club_registry(file_path):
    if not file_path.exists():
        print(f"❌ Error: {file_path} does not exist.")
        return

    # 1. Load the existing master file
    df = pd.read_csv(file_path)
    
    # 2. Data Normalization for Comparison
    # We create a temporary column to find matches regardless of casing or extra spaces
    df['NormName'] = df['Name'].astype(str).str.strip().str.upper()
    
    # 3. Identify Placeholder Records
    df['IsPlaceholder'] = df['ClubId'] < 0
    
    # 4. Identify Potential Name Duplicates
    # Find names that appear more than once across the entire registry
    name_counts = df.groupby('NormName')['ClubId'].transform('count')
    df['MultipleIDsFound'] = name_counts > 1
    
    # 5. Generate Recommendation Logic
    def get_recommendation(row):
        if row['IsPlaceholder'] and row['MultipleIDsFound']:
            return "REPLACE: Placeholder ID exists for a verified name"
        elif row['IsPlaceholder']:
            return "VERIFY: Placeholder ID with no verified match found"
        elif row['MultipleIDsFound']:
            return "REVIEW: Multiple verified IDs found for same name"
        else:
            return "VALID: Unique verified record"

    df['ActionRecommendation'] = df.apply(get_recommendation, axis=1)
    
    # 6. Sort for easier human review (Group duplicates together)
    df_audit = df.sort_values(by=['NormName', 'ClubId']).reset_index(drop=True)
    
    # Remove the temporary normalization column before saving
    df_audit = df_audit.drop(columns=['NormName'])
    
    # 7. Save to temporary audit file
    df_audit.to_csv(output_audit_file, index=False)
    
    print(f"✅ Audit Complete.")
    print(f"📍 Report saved to: {output_audit_file}")
    
    # Print high-level stats for the mission update
    print("\n--- Mission Intelligence Summary ---")
    print(f"Total Club Records: {len(df)}")
    print(f"Placeholder IDs (<0): {df['IsPlaceholder'].sum()}")
    print(f"Records requiring review/merge: {df['MultipleIDsFound'].sum()}")

# Execute Audit
if __name__ == "__main__":
    audit_club_registry(input_file)
